In [1]:
!pip install pyspark pandas requests sqlalchemy pymysql matplotlib


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: C:\Users\Lenovo\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
     ---------------------------------------- 0.0/52.8 kB ? eta -:--:--
     ------- -------------------------------- 10.2/52.8 kB ? eta -:--:--
     ------------------------------------ - 51.2/52.8 kB 890.4 kB/s eta 0:00:01
     -------------------------------------- 52.8/52.8 kB 926.0 kB/s eta 0:00:00
     ---------------------------------------- 0.0/41.7 kB ? eta -:--:--
     ---------------------------------------- 41.7/41.7 kB 2.0 MB/s eta 0:00:00
  Using cached idna-3.15-py3-none-any.whl.metadata (7.7 kB)
  Using cached certifi-2026.5.20-py3-none-any.whl.metadata (2.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
     ---------------------------------------- 0.0/121.0 kB ? eta -:--:--
     -------------------------------------- 121.0/121.0 kB 3.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.9 MB 7.7 MB/

In [2]:
import requests
import json
import pandas as pd
import matplotlib.pyplot as plt

from sqlalchemy import create_engine
from pyspark.sql import SparkSession

In [6]:
spark = SparkSession.builder \
    .appName("VideoGames-MultiSource-ETL") \
    .getOrCreate()

In [7]:
API_KEY = "2ec9b9a5188841d59822585a112d749f"

url = f"https://api.rawg.io/api/games?key={API_KEY}"

response = requests.get(url)

api_data = response.json()

In [8]:
api_games = api_data["results"]

api_df = pd.DataFrame(api_games)

api_df = api_df[
    [
        "id",
        "name",
        "released",
        "rating",
        "metacritic"
    ]
]

api_df.head()

,id,name,released,rating,metacritic
0,3498,Grand Theft Auto V,2013-09-17,4.47,92
1,3328,The Witcher 3: Wild Hunt,2015-05-18,4.64,92
2,4200,Portal 2,2011-04-18,4.58,95
3,4291,Counter-Strike: Global Offensive,2012-08-21,3.57,81
4,5286,Tomb Raider,2013-03-05,4.06,86


In [9]:
spark_api_df = spark.createDataFrame(api_df)

spark_api_df.show(5)

+----+--------------------+----------+------+----------+
|  id|                name|  released|rating|metacritic|
+----+--------------------+----------+------+----------+
|3498|  Grand Theft Auto V|2013-09-17|  4.47|        92|
|3328|The Witcher 3: Wi...|2015-05-18|  4.64|        92|
|4200|            Portal 2|2011-04-18|  4.58|        95|
|4291|Counter-Strike: G...|2012-08-21|  3.57|        81|
|5286|         Tomb Raider|2013-03-05|  4.06|        86|
+----+--------------------+----------+------+----------+
only showing top 5 rows


In [ ]:
from sqlalchemy import create_engine
import pandas as pd

engine = create_engine(
    "http://127.0.0.1/phpmyadmin/index.php?route=/table/sql&db=historical_events_db&table=historical_events"
)

with engine.connect() as connection:
    df = pd.read_sql(
        "SELECT * FROM historical_events",
        connection
    )

print(df)

OperationalError: (pymysql.err.OperationalError) (1045, "Access denied for user 'root'@'localhost' (using password: NO)")
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [ ]:
spark_reviews_df = spark.createDataFrame(reviews_df)

spark_reviews_df.show()

In [ ]:
files_df = pd.read_csv(
    "../data/files/local_games.csv"
)

files_df.head()

In [ ]:
spark_files_df = spark.createDataFrame(files_df)

spark_files_df.show()

In [ ]:
joined_df = spark_api_df.join(
    spark_reviews_df,
    spark_api_df.name == spark_reviews_df.game_name,
    "inner"
)

joined_df.show()

In [ ]:
api_df["rating"].hist()

plt.title("Distribución de ratings")

plt.show()

In [ ]:
joined_df.write.mode("overwrite").parquet(
    "../data/processed/games_parquet"
)